# Optimizer Showdown: SGD vs Momentum vs Adam

In this notebook we train the **same model architecture** on MNIST with three different optimizers and compare their convergence behavior.

**Objectives:**
- Understand how optimizer choice affects training speed and final accuracy
- Compare SGD, SGD + Momentum, and Adam on a real dataset
- Visualize training curves side by side

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")

## 1. Load and Prepare MNIST

In [ ]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize to [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten 28x28 -> 784
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print(f"Training samples: {x_train.shape[0]}")
print(f"Test samples: {x_test.shape[0]}")

## 2. Define Model Builder

We use a function so we get a **fresh, untrained** model for each optimizer.

In [ ]:
def build_model():
    """Simple 2-hidden-layer model for MNIST."""
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(784,)),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

EPOCHS = 10
BATCH_SIZE = 128

## 3. Train with Plain SGD (lr=0.01)

In [ ]:
model_sgd = build_model()
model_sgd.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_sgd = model_sgd.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_test, y_test),
    verbose=1
)

print(f"\nSGD final test accuracy: {history_sgd.history['val_accuracy'][-1]:.4f}")

## 4. Train with SGD + Momentum (lr=0.01, momentum=0.9)

In [ ]:
model_momentum = build_model()
model_momentum.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_momentum = model_momentum.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_test, y_test),
    verbose=1
)

print(f"\nSGD+Momentum final test accuracy: {history_momentum.history['val_accuracy'][-1]:.4f}")

## 5. Train with Adam (lr=0.001)

In [ ]:
model_adam = build_model()
model_adam.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_adam = model_adam.fit(
    x_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(x_test, y_test),
    verbose=1
)

print(f"\nAdam final test accuracy: {history_adam.history['val_accuracy'][-1]:.4f}")

## 6. Plot Training Accuracy Curves

In [ ]:
plt.figure(figsize=(12, 5))

# --- Accuracy ---
plt.subplot(1, 2, 1)
plt.plot(history_sgd.history['val_accuracy'], label='SGD', marker='o')
plt.plot(history_momentum.history['val_accuracy'], label='SGD + Momentum', marker='s')
plt.plot(history_adam.history['val_accuracy'], label='Adam', marker='^')
plt.title('Validation Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# --- Loss ---
plt.subplot(1, 2, 2)
plt.plot(history_sgd.history['val_loss'], label='SGD', marker='o')
plt.plot(history_momentum.history['val_loss'], label='SGD + Momentum', marker='s')
plt.plot(history_adam.history['val_loss'], label='Adam', marker='^')
plt.title('Validation Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 7. Comparison Table

In [ ]:
import pandas as pd

# Find epoch where each optimizer first crosses 95% val accuracy (convergence speed proxy)
def first_epoch_above(history, threshold=0.95):
    for i, acc in enumerate(history['val_accuracy']):
        if acc >= threshold:
            return i + 1  # 1-indexed
    return ">" + str(EPOCHS)

results = pd.DataFrame({
    'Optimizer': ['SGD (lr=0.01)', 'SGD + Momentum (0.9)', 'Adam (lr=0.001)'],
    'Final Val Accuracy': [
        f"{history_sgd.history['val_accuracy'][-1]:.4f}",
        f"{history_momentum.history['val_accuracy'][-1]:.4f}",
        f"{history_adam.history['val_accuracy'][-1]:.4f}"
    ],
    'Final Val Loss': [
        f"{history_sgd.history['val_loss'][-1]:.4f}",
        f"{history_momentum.history['val_loss'][-1]:.4f}",
        f"{history_adam.history['val_loss'][-1]:.4f}"
    ],
    'Epoch to 95% Acc': [
        first_epoch_above(history_sgd.history),
        first_epoch_above(history_momentum.history),
        first_epoch_above(history_adam.history)
    ]
})

results

## 8. Your Analysis

### TODO: Answer the following questions (write in the markdown cell below)

1. Which optimizer achieved the highest final accuracy?
2. Which optimizer converged the fastest (reached 95% accuracy soonest)?
3. Why is Adam so popular as a default optimizer? What advantages does it have over plain SGD?
4. Can you think of a scenario where plain SGD might actually be preferred?

**Your answers here:**

1. ...
2. ...
3. ...
4. ...